# Solutions — Vectorized Transformations, String Ops, Binning, np.select, Sessionization

Exercise notebook: `09_pandas/03_transformations_vectorized.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
pd.set_option("display.width", 140)
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
orders_raw = pd.read_csv(DATA / "csv" / "orders.csv", dtype=str, keep_default_na=False)
print(orders_raw.shape); orders_raw.head()

## Exercise 1 — 🟢 Easy · Arithmetic columns with assign

**Working solution**

In [ ]:
def add_amounts(df, tax_rate=0.18):
    return df.assign(amount=lambda d: d["quantity"] * d["unit_price"],
                     tax=lambda d: (d["amount"] * tax_rate).round(2),
                     total=lambda d: d["amount"] + d["tax"])

**Explanation**

Lambdas inside `assign` receive the frame *being built*, so later columns can use earlier ones. No loops, no mutation.

**Complexity:** O(n), vectorized.

**Common mistakes**

- `df['amount'] = ...` mutates the caller's frame.
- `df.apply(lambda r: r.quantity * r.unit_price, axis=1)` — ~100× slower.

## Exercise 2 — 🟡 Medium · Vectorized string cleanup

**Working solution**

In [ ]:
def clean_text_columns(df):
    out = df.copy()
    for col, method in (("channel", "lower"), ("status", "upper"), ("country", "upper")):
        if col in out:
            out[col] = getattr(out[col].astype("string").str.strip().str, method)()
    if "name" in out:
        out["name"] = out["name"].astype("string").str.split().str.join(" ").str.title()
    cols = [c for c in ("channel", "status", "country", "name") if c in out]
    out[cols] = out[cols].replace("", pd.NA)
    return out

**Explanation**

`.str` methods apply to every element and propagate missing values. `.str.split().str.join(' ')` is the vectorized whitespace collapse. The `string` dtype uses `pd.NA` for missing values.

**Complexity:** O(total characters).

**Common mistakes**

- `df['x'].apply(lambda s: s.strip())` crashes on NaN.

## Exercise 3 — 🟡 Medium · Binning with pd.cut

**Working solution**

In [ ]:
import numpy as np
import pandas as pd

def add_buckets(df):
    return df.assign(amount_bucket=pd.cut(df["amount"], bins=[0, 1000, 10_000, np.inf], right=False, labels=["<1k", "1k-10k", "10k+"]),
                     is_large=df["amount"] >= 10_000)

**Explanation**

`pd.cut` produces an ordered categorical — memory-efficient and sorts in bucket order, not alphabetically.

**Complexity:** O(n log bins).

**Common mistakes**

- Default `right=True` puts exactly 1000 into '<1k'.

## Exercise 4 — 🟠 Hard · if/elif logic with np.select (and a benchmark)

**Working solution**

In [ ]:
import numpy as np
import pandas as pd

def risk_level(df):
    foreign = df["country"].str.upper() != df["home_country"].str.upper()
    failed = df["failed_attempts"].fillna(0) if "failed_attempts" in df else 0
    conditions = [
        (df["amount"] > 200_000) | (failed >= 3),
        foreign & (df["amount"] > 50_000),
        df["hour"].between(0, 4) & (df["amount"] > 10_000),
        (df["currency"].str.upper() != "INR") | foreign,
    ]
    return pd.Series(np.select(conditions, ["HIGH", "HIGH", "MEDIUM", "MEDIUM"], default="LOW"), index=df.index, name="risk")

**Explanation**

`np.select` evaluates boolean condition columns and picks the first matching choice per row — a vectorized `if/elif/else` (SQL `CASE WHEN`). On 100k rows it's typically
50–200× faster than `apply(axis=1)`, which calls a Python function per row.

**Benchmark scaffold**

In [ ]:
import time
import numpy as np
import pandas as pd
rng = np.random.default_rng(0)
n = 100_000
big = pd.DataFrame({"amount": rng.uniform(0, 300_000, n), "currency": rng.choice(["INR", "USD"], n), "hour": rng.integers(0, 24, n),
                    "country": rng.choice(["IN", "US"], n), "home_country": "IN", "failed_attempts": rng.integers(0, 4, n)})

def row_risk(r):
    foreign = r.country.upper() != r.home_country.upper()
    if r.amount > 200_000 or r.failed_attempts >= 3: return "HIGH"
    if foreign and r.amount > 50_000: return "HIGH"
    if 0 <= r.hour < 5 and r.amount > 10_000: return "MEDIUM"
    if r.currency.upper() != "INR" or foreign: return "MEDIUM"
    return "LOW"

t0 = time.perf_counter(); slow = big.apply(row_risk, axis=1); t1 = time.perf_counter(); fast = risk_level(big); t2 = time.perf_counter()
print(f"apply: {t1 - t0:.2f}s  np.select: {t2 - t1:.4f}s  same: {(slow == fast).all()}")

**Complexity:** O(n × rules), vectorized.

**Common mistakes**

- Using `np.where` nested 4 levels deep — unreadable.
- Forgetting NaN in `failed_attempts`.

## Challenge — 🔴 Very Hard · Vectorized sessionization

**Working solution**

In [ ]:
import pandas as pd

def sessionize(events, gap_minutes=30):
    df = events.sort_values(["user_id", "ts"], kind="stable").copy()
    gap = df.groupby("user_id")["ts"].diff()
    new_session = gap.isna() | (gap > pd.Timedelta(minutes=gap_minutes))
    df["session_no"] = new_session.astype(int).groupby(df["user_id"]).cumsum()
    df["session_id"] = df["user_id"].astype(str) + "-" + df["session_no"].astype(str)
    stats = (df.groupby("session_id")
               .agg(user_id=("user_id", "first"), start=("ts", "min"), end=("ts", "max"), events=("ts", "size"))
               .reset_index())
    stats["duration_min"] = (stats["end"] - stats["start"]).dt.total_seconds() / 60
    df = df.drop(columns="session_no")
    df.attrs["sessions"] = stats.sort_values("session_id").reset_index(drop=True)
    return df

**Explanation**

The 'gap → flag → cumulative sum' trick turns a stateful loop into three vectorized steps: `diff` per group, a boolean 'starts a new session' column, and `cumsum` per group to
number sessions. The same pattern in Spark uses `lag()` + `sum().over(window)`.

**Complexity:** O(n log n) for the sort; the rest O(n).

**Common mistakes**

- Looping per user with `iterrows`.
- `diff()` without `groupby` — gaps leak across users.